## Pandas Filtering, GroupBy, and Aggregation
This notebook covers essential operations in pandas for data manipulation and analysis, including filtering, grouping, and aggregation. These concepts are crucial for exploratory data analysis and feature engineering in machine learning.

### Importing Libraries and Creating a Sample Dataset
We start by importing pandas and numpy, then create a sample dataset for demonstration purposes.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Department': ['Sales', 'IT', 'Sales', 'HR', 'IT', 'Sales', 'HR'],
    'Employee': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace'],
    'Salary': [50000, 70000, 55000, 48000, 75000, 52000, 49000],
    'Performance': [8.5, 9.0, 7.5, 8.0, 9.2, 8.3, 7.8],
    'YearsAtCompany': [2, 5, 3, 1, 6, 2, 1]
})

print("Sample Employee DataFrame:")
print(df)
print("\n" + "="*60)

### Filtering (Boolean Indexing)
Filtering allows us to create subsets of data based on conditions. This is essential for data exploration and splitting data into training and testing sets.

In [ ]:
# Single condition
salespeople = df[df['Department'] == 'Sales']
print("\nAll Salespeople:")
print(salespeople)

# Greater than / Less than
high_performers = df[df['Performance'] > 8.5]
print("\nHigh performers (Performance > 8.5):")
print(high_performers)

# Multiple conditions (AND)
senior_sales = df[(df['Department'] == 'Sales') & (df['YearsAtCompany'] >= 2)]
print("\nSenior sales employees (2+ years):")
print(senior_sales)

# Multiple conditions (OR)
high_earners = df[(df['Salary'] > 70000) | (df['Performance'] > 9.0)]
print("\nHigh earners OR top performers:")
print(high_earners)

# Negation (NOT)
non_hr = df[df['Department'] != 'HR']
low_performance = df[~(df['Performance'] > 8.0)]  # NOT Performance > 8.0
print("\nNon-HR employees:")
print(non_hr)

# Using isin for multiple values
tech_staff = df[df['Department'].isin(['IT', 'HR'])]
print("\nIT and HR employees:")
print(tech_staff)

# Using str methods for string filtering
names_with_a = df[df['Employee'].str.startswith('A')]
print("\nEmployees with names starting with 'A':")
print(names_with_a)

The output of these filtering operations are subsets of the original DataFrame, each containing rows that match the specified conditions. A common mistake is forgetting to assign the result back to a variable or using it directly in further operations. A quick tip is to always verify the shape and content of the resulting DataFrame to ensure the filtering was applied as expected.

### GroupBy: Split-Apply-Combine
GroupBy operations allow us to split the data into groups based on some criteria, apply a function to each group, and then combine the results.

In [ ]:
# Group by single column and compute aggregate
groupby_dept = df.groupby('Department')
print("\n" + "="*60)
print("\nGroupBy Department - Average Salary:")
avg_salary = df.groupby('Department')['Salary'].mean()
print(avg_salary)

# Multiple aggregations
dept_stats = df.groupby('Department')[['Salary', 'Performance']].agg(['mean', 'std', 'min', 'max'])
print("\nDepartment Statistics (Mean, Std, Min, Max):")
print(dept_stats)

# Custom aggregation names
dept_summary = df.groupby('Department')['Salary'].agg(['mean', 'count']).rename(
    columns={'mean': 'AvgSalary', 'count': 'Count'}
)
print("\nDepartment Summary (custom names):")
print(dept_summary)

# Group by multiple columns
df['PerfLevel'] = pd.cut(df['Performance'], bins=[0, 8.0, 10.0], labels=['Standard', 'Excellent'])
multi_group = df.groupby(['Department', 'PerfLevel'])[['Salary']].mean()
print("\nSalary by Department and Performance Level:")
print(multi_group)

# Count occurrences per group
dept_counts = df['Department'].value_counts()
print("\nEmployee counts by department:")
print(dept_counts)

The output of GroupBy operations can vary depending on the aggregation function used. A common mistake is not specifying the column(s) to aggregate, leading to an error. A quick tip is to use the `describe` method for a quick summary of central tendency and dispersion for numeric columns.

### Agg: Apply Multiple Functions
The `agg` function allows us to apply multiple aggregation functions to different columns or the same column.

In [ ]:
# Different functions for different columns
agg_dict = {
    'Salary': ['mean', 'std'],
    'Performance': ['mean', 'max'],
    'YearsAtCompany': 'median'
}
result = df.groupby('Department').agg(agg_dict)
print("\n" + "="*60)
print("\nCustom aggregation by department:")
print(result)

# Using lambda for custom aggregations
custom_agg = df.groupby('Department').agg({
    'Salary': ['min', 'max', lambda x: x.max() - x.min()],
    'Performance': lambda x: x.sum() / len(x)
})
print("\nCustom aggregation (with lambda):")
print(custom_agg)

The output of `agg` operations provides a flexible way to compute multiple statistics. A common mistake is not correctly defining the aggregation dictionary. A quick tip is to use named aggregation for clarity when applying multiple functions to a single column.

### Transform: Group-wise Transformations
The `transform` function applies a function to each group and returns a result that has the same shape as the original DataFrame.

In [ ]:
# Standardize salary within each department
df['Salary_Standardized'] = df.groupby('Department')['Salary'].transform(
    lambda x: (x - x.mean()) / x.std()
)
print("\n" + "="*60)
print("\nSalary standardized within departments:")
print(df[['Department', 'Salary', 'Salary_Standardized']])

# Rank within groups
df['Dept_SalaryRank'] = df.groupby('Department')['Salary'].transform('rank')
print("\nSalary rank within departments:")
print(df[['Department', 'Salary', 'Dept_SalaryRank']])

The output of `transform` operations is a new Series or DataFrame with the same shape as the original data, allowing for broadcasting back to the original DataFrame. A common mistake is not understanding how the transformation affects the data. A quick tip is to verify the transformation by comparing the original and transformed values.

### Filtering Groups
Filtering groups based on aggregate conditions allows us to select entire groups that meet certain criteria.

In [ ]:
# Filter groups by aggregate condition
high_avg_depts = df.groupby('Department').filter(lambda x: x['Salary'].mean() > 55000)
print("\n" + "="*60)
print("\nEmployees in high-paying departments (avg > 55000):")
print(high_avg_depts)

# Keep groups with sufficient size
large_depts = df.groupby('Department').filter(lambda x: len(x) >= 2)
print("\nDepartments with 2+ employees:")
print(large_depts)

The output of filtering groups shows only the groups that meet the specified condition. A common mistake is applying the filter incorrectly, leading to unexpected results. A quick tip is to test the filter condition on a small subset of data to ensure it works as expected.

### Practical ML Scenarios
These operations are essential in real-world machine learning scenarios for feature engineering, data exploration, and model preparation.

In [ ]:
# Scenario 1: Feature engineering - department averages as features
dept_avg = df.groupby('Department')['Salary'].transform('mean').rename(
    'Dept_AvgSalary'
)
X_with_feature = pd.concat([df[['Salary', 'Performance']], dept_avg], axis=1)
print("With department average salary feature:")
print(X_with_feature.head())

# Scenario 2: Identify outliers within groups
def identify_outliers(group):
    Q1 = group.quantile(0.25)
    Q3 = group.quantile(0.75)
    IQR = Q3 - Q1
    return (group < Q1 - 1.5*IQR) | (group > Q3 + 1.5*IQR)

df['SalaryOutlier'] = df.groupby('Department')['Salary'].transform(identify_outliers)
print("\nOutliers by department (using IQR):")
print(df[['Department', 'Salary', 'SalaryOutlier']])

# Scenario 3: Stratified sampling for train/test split
train_mask = df.groupby('Department').apply(lambda x: np.random.choice([True, False], size=len(x), p=[0.7, 0.3])).reset_index(level=0, drop=True)
train_set = df[train_mask]
test_set = df[~train_mask]
print("\nStratified train/test split:")
print(f"Train: {len(train_set)}, Test: {len(test_set)}")
print(f"Dept distribution in train: {train_set['Department'].value_counts().to_dict()}")

## Key Takeaways
* Use boolean indexing for filtering data based on conditions.
* Apply GroupBy operations for splitting data, applying functions, and combining results.
* Utilize `agg` for computing multiple statistics and `transform` for group-wise transformations.
* Filter groups based on aggregate conditions for selecting entire groups that meet certain criteria.
* These operations are foundational for exploratory data analysis and feature engineering in machine learning.